# 第18課：使用加密收據保障 AI 代理

## 實作筆記本

本筆記本包含四個練習：

1. <strong>簽署您的第一張收據</strong> 用於代理工具呼叫並驗證它。  
2. <strong>篡改收據</strong> 並觀察驗證失敗。  
3. <strong>建立三張收據鏈</strong> 並確認鏈條完整性。  
4. **包裝 Microsoft Agent Framework 工具呼叫** 使每個操作都發出收據。

所有加密基元皆從維護良好的函式庫匯入（使用 `pynacl` 的 Ed25519，`jcs` 用於 RFC 8785 正規化 JSON，Python 標準函式庫 `hashlib` 提供 SHA-256）。收據邏輯本身是容易閱讀和修改的純 Python。

請依序執行各個代碼區塊。每個章節簡短且自成一格。


## Setup

安裝兩個依賴。兩者皆為寬鬆授權（Apache-2.0 / MIT）。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## 助手工具

這兩個助手負責處理 base64url 編碼（無填充）及對任意物件進行 SHA-256 雜湊。它們讓筆記本的其餘部分專注於收據邏輯本身。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: 簽署你的第一張收據

想像一下，我們的 **Contoso Travel** 代理剛為客戶查詢了從悉尼到洛杉磯的航班。我們想將這次工具調用記錄為已簽署的收據，讓未來的審計員能在不必信任我們的情況下進行驗證。

### Step 1.1: 產生簽署金鑰

在生產環境中，代理的簽署金鑰會保存在硬體安全模組 (HSM)、Azure Key Vault 或類似的受保護儲存中。這堂課我們先在記憶體中產生一把新的金鑰。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: 建立收據的負載

負載包含我們希望收據能證明的一切：誰執行了操作、使用了什麼工具、帶了哪些參數、返回了什麼、遵守了哪條政策，以及何時進行。我們對參數和結果進行哈希，而不是直接包含它們，這樣收據就不會洩露敏感內容。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: 簽署及組裝收據

三個步驟：

1. 使用 JCS 對 payload 進行規範化，使兩個實作產生相同邏輯的收據時，得到位元相同的位元組。
2. 使用 SHA-256 對規範化後的位元組進行雜湊。
3. 使用 Ed25519 私鑰對雜湊值進行簽署。

然後，將簽名附加到原始 payload 上，以生成最終收據。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: 驗證收據

驗證是反向進行的過程。我們去除簽名，重新計算規範哈希，並使用收據中的公鑰來檢查簽名。

進行此驗證的審核員只需要收據本身。無需呼叫任何服務，無需查詢任何金鑰目錄，亦無需信任。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

你應該會看到 `Receipt is valid: True`。代理已經產生了它第一個加密簽名的審計記錄。


## 第 2 節：篡改收據

收據的整個重點是讓篡改一目了然。讓我們來證明這一點。

我們將修改收據中的一個字元，然後觀察驗證失敗。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 發生了什麼事？

當我們更改了 `policy_id`，標準化的位元組也隨之改變。這些位元組的 SHA-256 雜湊也改變了。簽名（它是基於原始雜湊產生的）不再符合新的雜湊。驗證正確地返回了 `False`。

除非攻擊者擁有私鑰，否則無法修改收據的任何欄位並仍能通過驗證。只要私鑰存放在金鑰庫中並且公鑰已公開，篡改是不可能被隱藏的。

自己試試看：在上面的儲存格中修改 `tool_name`、`agent_id` 或 `timestamp`，然後重新執行。每一次修改都會產生無效的收據。


## Section 3: 連鎖收據

一張收據保護一個操作。大多數代理人會執行多個操作。為了使整個序列可防篡改，我們通過在新收據的有效負載中包含前一張收據的哈希，將每張收據連接到前一張收據。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

如果有人刪除或重新排序了收據，鏈條就會在恰好那個點中斷。任何後續收據的驗證都會失敗，因為它的 `previous_receipt_hash` 不再匹配其前一個的實際哈希。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

現在通過篡改中間的收據來斷開鏈條，然後重新驗證。被篡改的收據無法通過其簽名檢查，並且下一張收據無法通過其鏈接檢查（因為其 `previous_receipt_hash` 不再匹配被修改的中間收據的哈希）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

收據 0 仍然通過驗證（它沒有被修改且沒有前置收據需要依賴）。收據 1 由於我們修改了 `tool_args_hash`，簽名檢查失敗。收據 2 的鏈接檢查失敗，因為它的 `previous_receipt_hash` 是根據原始（現在已被修改的）收據 1 計算的。

即使攻擊者重新簽署被修改的收據 1（而他們沒有私鑰無法做到這點），收據 2 的鏈接不匹配仍會暴露篡改痕跡。為了隱藏改動，攻擊者必須重新簽署從修改點之後的每一張收據，這就需要擁有私鑰。


## Section 4: 用收據簽署包裝代理工具調用

在實際部署中，您不希望每個代理編寫者都記得調用 `make_receipt`。您希望每次調用工具時，收據簽署都是自動的。

這裡是最簡單的範例：一個包裝類，它接受任何可調用的工具函數並返回一個發出收據的版本。這適用於任何代理框架，包括 Microsoft Agent Framework（`agent_framework.azure`）。

如果您還沒有設置 Azure AI Foundry 專案，下面的本地模擬仍然展示了這種模式。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### 與 Microsoft Agent Framework 整合

上述的 `ReceiptedTool` 包裝器是不依賴框架的。要在使用 Microsoft Agent Framework 建立的代理中使用它，請將包裝過的函數註冊為工具。以下是一個草稿（你可以用真實的 Azure AI Foundry 工具註冊取代示範）：

```python
# 顯示整合形態的偽代碼。
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="你係 Contoso 旅遊代理 ...",
#     tools=[receipted_lookup],   # 包裝咗嘅工具，唔係原始函數
# )
# response = agent.run("搵六月由悉尼去洛杉磯嘅航班。")
#
# # 運行完之後，代理使用嘅每個工具調用都有簽署收據：
# audit_chain = receipted_lookup.receipts
```

代理框架不需要知道有關收據的任何資訊。收據簽名是在工具周圍包裝的，而不是強行整合到框架中。這就是如何在不重寫代理的情況下，為現有的代理代碼新增來源追蹤。


## 回顧與進階挑戰

你已經：

- 生成了一組 Ed25519 金鑰對。
- 建立並簽署了一個代理工具呼叫的收據。
- 僅使用公鑰離線驗證收據。
- 篡改過收據並觀察驗證失敗。
- 建立了一串包含三個收據的雜湊鏈序列。
- 篡改鏈中間部分並觀察簽名失敗與鏈結失敗。
- 用自動簽署收據包裝了一個工具函式。

**進階挑戰。** 擴展收據架構，新增一個 `request_id` 欄位（用於分散式追蹤的 UUID）。更新 `make_receipt` 以包含該欄位，並確認收據仍可端到端驗證。然後在簽署後修改該欄位並確認驗證失敗。這會讓你深入理解每一個位元組的規範編碼如何影響簽名。

**重要界限。** 收據只證明三件事，而且只有三件事：歸屬（此金鑰簽署了此內容）、完整性（內容自簽署後未被更改）與排序（此收據是在那張收據之後產生）。它們不證明代理的行為是正確的、`policy_id` 中指定的政策是否真的被評估過、或代理是否遵守了每條規則。收據是基礎。治理則是你建立在其上的系統。

請再次閱讀課程 README，並以此界限為重點。團隊最常犯的錯誤是認為「我們有收據」即可代表「我們有治理」。事實並非如此。收據讓代理行為可被審核，卻不保證其正確性。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
